# 데이터 전처리

우리가 실제로 접하는 데이터는 사이킷 런에서 제공하는 데이터와 다르게, 
머신러닝을 적용하기 전에 데이터를 가공해야 하는 경우가 많습니다.

<span style="color:red">**이를 데이터 전처리, 피처 엔지니어링**</span> 등으로 부릅니다.

## 전처리과정

1. 결측치 처리
2. 클래스 라벨 설정
3. 원-핫 인코딩
4. 데이터 스케일링

    

### 결측치 처리

데이터 셋의 일부에 데이터가 존재하지 앟는 경우 해당 데이터를 <span style="color:blue">결측치(missing value)</span> 이라고 부르며,

머신러닝 알고리즘을 적용하기 전에 미리 결측치 처리를 하는 것이 중요합니다.

우선 실습을 위해 결측치를 포함하는 데이터를 생성해 봅니다.

In [66]:
# 데이터 프레임 생성을 위한 라이브러리 불러오기

import numpy as np
import pandas as pd

In [67]:
# pd.DataFrame 을 통해 직접 데이터 프레임을 생성할 수 있음

df = pd.DataFrame([
    [42, 'male', 12, 'reading', 'class2'],
    [35, 'unknown', 3, 'cooking', 'class1'],
    [1000, 'female', 7, 'cycling', 'class3'],
    [1000, 'unknown', 21, 'unknown', 'unknown'],

])

#df. columns 를 통해 열 이름을 정할 수 있음

df.columns = ['age',
              'gender',
              'month_birth',
              'hobby', 'target']

df

,age,gender,month_birth,hobby,target
0,42,male,12,reading,class2
1,35,unknown,3,cooking,class1
2,1000,female,7,cycling,class3
3,1000,unknown,21,unknown,unknown


앞서 <span style='color:red'>결측치</span> 는 <mark>데이터가 존재하지 않는 것</mark>이라고 했습니다. 

하지만 위에 데이터 프레임을 보면 데이터 값이 모두 존재하지만, 부적절한 값들이 존재합니다.

*(나이가 1000, 혹은 unknown)*

이런 값들은 부적절한 결측치로 처리하는 것이 좋습니다.

데이터 셋에 존재하는 부적절한 값들을 결측치로 변경해 봅시다.

  결측치 변경에 앞서, 각 열(column) 에 어떤 값이 존재하는지 파악해야 합니다.

데이터 셋들이 작을 경우 한눈에 볼 수 있지만, 일반적으로 데이터 셋들은 크기가 커서 한번에 파악하기 어려우므로

다음과 같이 <span style="color:blue"> 각 열의 유니크 값들을 확인해 봅니다</span>

*유니크 값: 중복을 제외한 값*

In [68]:
#각 열의 유니크 값을 파악하기 위해 unique()메소드 확인

df['age'].unique()

array([  42,   35, 1000])

In [69]:
print(df['gender'].unique())
print(df['month_birth'].unique())
print(df['hobby'].unique())
print(df['target'].unique())

['male' 'unknown' 'female']
[12  3  7 21]
['reading' 'cooking' 'cycling' 'unknown']
['class2' 'class1' 'class3' 'unknown']


나이와 마찬가지로 나머지 열의 값을 파악하여 부적절한 값이 있는지 확인해 봅니다.

`gender` : `unknown` 이라는 부적절한 값이 보입니다.  
`month_birth` : 1부터 12 까지 있으므로 21 은 부적절합니다.  
`hobby` : `unknown` 이라는 부적절한 값이 보입니다.  
`target` : 마찬가지로 `unknown` 이라는 부적절한 값이 보입니다.  

<span style = "color:blue">**feature 별로 부적절한 값을 확인했으므로 이들을 결측치로 바꿉니다.**

`loc` : pandas에서 행과 열을 이름(label) 기준으로 선택하는 기능

In [70]:
#부적절한 값들을 결측치로 바뀌기 위해 np.nan 사용

df.loc[df['age'] > 150, ['age']] = np.nan
df.loc[df['gender'] =="unknown" , ['gender']] = np.nan
df.loc[df['month_birth'] > 12, ['month_birth']] = np.nan
df.loc[df['hobby'] =='unknown', ['hobby']] = np.nan
df.loc[df['target'] == 'unknown', ['target']] = np.nan


부적절한 값들을 처리한 데이터 셋들은 다음과 같습니다

In [71]:
df

,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,NaN,3.0,cooking,class1
2,NaN,female,7.0,cycling,class3
3,NaN,NaN,NaN,NaN,NaN


결측치 처리의 첫 단계로 부적절한 값을 결측치로 변환하였습니다.  
<span style = "color:blue">**결측치를 포함하는 데이터는 머신러닝에 적합하지 않는 경우가 많습니다.**

데이터 셋을 처음 접했을때 해야 하는 일은,  
<mark>**각 열에 결측치가 존재하는지 확인하는 것 입니다.** </mark>

`df.isnull().sum()` : 각 열의 결측치 개수 확인 가능

In [72]:
df.isnull().sum()

age            2
gender         2
month_birth    1
hobby          1
target         1
dtype: int64

결측치를 처리하는 데 주로 쓰이는 방법은  

결측치를 <span style = "color:blue"> **삭제하거나 다른 값으로 변경합니다.** </span>

`dropna()` :  결측치 부분을 제거하는 함수

In [73]:
from IPython.display import display
# dropna 를 사용하여 결측치를 포함한 행 (row) (axis - 0) 삭제

df2 = df.dropna(axis = 0)

print("결측치 제거 전")
display(df)

print("결측치 존재 행 (row) 제거 후")
display(df2)

결측치 제거 전


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,NaN,3.0,cooking,class1
2,NaN,female,7.0,cycling,class3
3,NaN,NaN,NaN,NaN,NaN


결측치 존재 행 (row) 제거 후


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2


위와 같이 결측치를 포함한 행이 삭제되었습니다.

In [74]:
# 결측치를 포함한 열 ( column)(axis = 1) 삭제
df3 = df.dropna(axis = 1)

print("결측치 제거 전")
display(df)

print("결측치 존재 열(column) 제거 후")
display(df3)

결측치 제거 전


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,NaN,3.0,cooking,class1
2,NaN,female,7.0,cycling,class3
3,NaN,NaN,NaN,NaN,NaN


결측치 존재 열(column) 제거 후


""
0
1
2
3


위와 같이 결측치를 포함한 열이 삭제되었습니다.

In [75]:
#모든 값이 결측치인 행 삭제
#how = all 옵션을 사용하면 해당 행이 모두 결측치인 경우에만 행을 삭제

df4 = df.dropna(how = 'all')

print("결측치 제거 전")
display(df)

print("모든값이 결측치인 행 제거")
display(df4)

결측치 제거 전


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,NaN,3.0,cooking,class1
2,NaN,female,7.0,cycling,class3
3,NaN,NaN,NaN,NaN,NaN


모든값이 결측치인 행 제거


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,NaN,3.0,cooking,class1
2,NaN,female,7.0,cycling,class3


In [76]:
#값이 2개 미만인 행 삭제
#thresh = n 옵션은 각 행에 결측치를 제외한 값이 n개보다 적을 경우 삭제

df5 = df.dropna(thresh = 2)

print("결측치 제거 전")
display(df)

print("값이 2개 미만인 행 삭제")
display(df5)


결측치 제거 전


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,NaN,3.0,cooking,class1
2,NaN,female,7.0,cycling,class3
3,NaN,NaN,NaN,NaN,NaN


값이 2개 미만인 행 삭제


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,NaN,3.0,cooking,class1
2,NaN,female,7.0,cycling,class3


In [77]:
#특정 열에 결측치가 있는 경우 행 삭제
#
df6 = df.dropna(subset = ["gender"])

print("결측치 제거 전")
display(df)

print("gender 에 결측치가 있는경우 행 삭제")
display(df6)
                          

결측치 제거 전


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,NaN,3.0,cooking,class1
2,NaN,female,7.0,cycling,class3
3,NaN,NaN,NaN,NaN,NaN


gender 에 결측치가 있는경우 행 삭제


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
2,NaN,female,7.0,cycling,class3


결측치를 다른 값으로 변경해 봅니다.  
해당 열의 결측치를 바꾸고 싶은 값으로 `alter_values` 를 설정하고 설정한 값으로 변환합니다.  

필자는 `age` `month_birth` 는 0으로, 나머지 열은 U로 변환하겠습니다.  

<span style="color:blue">**필자처럼 결측치에 특정 값을 넣는 경우도 있고,  
해당 열의 평균값을 넣는 방법 등 여러가지 변경 방법이 존재합니다.**

In [78]:
#결측치 대체하기
alter_values = {
    'age' : 0,  
    'gender' : 'U',
    'month_birth' : 0,
    'hobby' : 'U',
    'target' : 'class4'
}

df7 = df.fillna(value = alter_values)
print("결측치 제거 전")
display(df)

print("결측치를 다른 값으로 변경")
display(df7)

결측치 제거 전


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,NaN,3.0,cooking,class1
2,NaN,female,7.0,cycling,class3
3,NaN,NaN,NaN,NaN,NaN


결측치를 다른 값으로 변경


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,U,3.0,cooking,class1
2,0.0,female,7.0,cycling,class3
3,0.0,U,0.0,U,class4


### 클래스 라벨 설정

클래스 라벨 설정에 대해 알아봅니다.  
여기서 <span style = "color : red ">라벨(label)</span>이란 이름표라고 생각하면 됩니다  

라벨을 붙이는 것을 <span style = "color : red ">라벨링(labeling)</span>이라고 합니다.  
<mark>이는 피처 데이터가 나타내는 것이 무엇인지 뜻합니다.</mark>

In [80]:
#라벨링 전 데이터
print("라벨링 전")
display(df7)

라벨링 전


,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,U,3.0,cooking,class1
2,0.0,female,7.0,cycling,class3
3,0.0,U,0.0,U,class4


`target` 열을 보면 문자열(string) 인데,   
데이터 전처리 과정에서는 문자열을 정수(int) 로 바꿔야 하는 경우가 있습니다.  

`sklearn` → 사이킷런이라는 라이브러리(패키지)  
`preprocessing` → 사이킷런 안의 전처리 모듈  
`LabelEncoder` → 그 모듈 안의 클래스  
`inverse_transform()` → `LabelEncoder` 객체가 가진 메서드


In [82]:
#클래스 라벨링을 위해 사이킷런의 preprocessing 패키지의 labelEncoder 함수를 사용

from sklearn.preprocessing import LabelEncoder

#활용할 데이터를 df8 이라고 함

df8 = df7

#LabelEncoder()를 이용해 라벨링 모형 설정

class_label = LabelEncoder()

#타깃 변수 값을 불러오기

data_value = df8['target'].values

#라벨링 값에 데이터 값을 넣고 변환

y_new = class_label.fit_transform(data_value)

#라벨링 된 결과 확인

y_new

array([1, 0, 2, 3])

In [83]:
df8['target'] = y_new
df8

,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,1
1,35.0,U,3.0,cooking,0
2,0.0,female,7.0,cycling,2
3,0.0,U,0.0,U,3


`target` 이 새로 라벨링이 된 것을 볼 수 있습니다.  

클래스 라벨링한 데이터를 원래대로 돌리고 싶다면 `inverse_transform` 메소드 사용  

In [85]:
y_ori = class_label.inverse_transform(y_new)
y_ori

array(['class2', 'class1', 'class3', 'class4'], dtype=object)

다시 문자 형태의 클래스 라벨을 확인 할 수 있습니다

In [87]:
df8["target"] = y_ori
df8

,age,gender,month_birth,hobby,target
0,42.0,male,12.0,reading,class2
1,35.0,U,3.0,cooking,class1
2,0.0,female,7.0,cycling,class3
3,0.0,U,0.0,U,class4


지금까지 사이킷런 라이브러리를 활용한 클래스 라벨링에 대해 알아보았습니다.  

이번에는 사이킷런을 사용하지 않고 직접 클래스 라벨링하는 방법을 알아봅니다.

In [89]:
#클래스 라벨링
#target 데이터 불러오기

y_arr = df8["target"].values

# sort() 는 오름차순으로 정렬하는 함수임

y_arr.sort()

y_arr

array(['class1', 'class2', 'class3', 'class4'], dtype=object)

In [91]:
num_y = 0
dic_y = {}

for ith_y in y_arr:
    dic_y[ith_y] = num_y
    num_y += 1

dic_y

{'class1': 0, 'class2': 1, 'class3': 2, 'class4': 3}